# Dagalog Jupyter Kernel — Introduction

This notebook demonstrates the **Dagalog** kernel for interactive RDF data pipelines.
Each cell is a pipeline step; the in-memory `Datastore` persists across all cells in a session.

## Setup

```bash
# Build and install the kernel (run from repo root)
cargo build -p dagalog-kernel
./target/debug/dagalog-kernel install

# Launch (from repo root so relative paths work)
jupyter lab notebooks/dagalog_intro.ipynb
```

Select **Dagalog (SPARQL + RDF)** as the kernel.

## Step 1 — Load inline Turtle

`%%turtle` parses the cell body as Turtle and adds the triples to the session datastore.

In [1]:
%%turtle
@prefix ex:   <http://example.com/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .

ex:Alice a foaf:Person ; foaf:name "Alice" ; foaf:age 30 ; ex:worksFor ex:Acme .
ex:Bob   a foaf:Person ; foaf:name "Bob"   ; foaf:age 25 ; ex:worksFor ex:Acme .
ex:Acme  a ex:Company  ; foaf:name "Acme Corp" .

Loaded 10 triples.

## Step 2 — Query with SPARQL SELECT

Cells without a `%%` prefix are treated as SPARQL. SELECT results render as an HTML table.

In [ ]:
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX ex:   <http://example.com/>

SELECT ?person ?name ?age WHERE {
    ?person a foaf:Person ;
            foaf:name ?name ;
            foaf:age  ?age .
}
ORDER BY ?name

## Step 3 — Apply an RML CSV mapping

`%%rml` loads an [RML mapping file](https://www.w3.org/TR/rml/) and runs it against
the declared CSV source, adding the resulting triples to the session datastore.

Paths are relative to the directory from which `jupyter lab` was launched
(the repo root when you follow the setup instructions above).

In [ ]:
%%rml tests/testdata/rml_persons_mapping.ttl

## Step 4 — OWL-RL Reasoning

`%%reason` runs OWL-RL materialisation on the current datastore, inferring new triples.

In [ ]:
%%reason

## Step 5 — Inspect the graph after loading

Query to see all people known to the datastore.

In [ ]:
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX ex:   <http://example.com/>

SELECT ?person ?name WHERE {
    ?person a foaf:Person ;
            foaf:name ?name .
}
ORDER BY ?name

## Step 6 — Inline Datalog rules

`%%datalog` parses and materialises forward-chaining Datalog rules.

In [ ]:
%%datalog
?x <http://example.com/colleague> ?y :-
    ?x <http://example.com/worksFor> ?org ,
    ?y <http://example.com/worksFor> ?org .

In [ ]:
PREFIX ex: <http://example.com/>

SELECT ?person ?colleague WHERE {
    ?person ex:colleague ?colleague .
    FILTER (?person != ?colleague)
}
ORDER BY ?person ?colleague